In [17]:
import pyarrow.parquet as pq

pq_path = r"parquet_files_dataset/compress/compress-gzip-all-events-run0.parquet"
start, stop = 1, 20  # rows [1000:1020]

pf = pq.ParquetFile(pq_path)
tbl = pf.read()                       # reads whole file (can be big)
df = tbl.to_pandas().iloc[start:stop] # slice
display(df)


,t_sec,dt_sec,cpu,event_id
1,31591.491350,5.360000e-07,1,33
2,31591.491351,8.080000e-07,1,34
3,31591.491351,6.320000e-07,2,23
4,31591.491352,5.100000e-07,1,20
5,31591.491352,3.780000e-07,2,18
6,31591.491352,3.150000e-07,2,18
7,31591.491353,2.910000e-07,2,25
8,31591.491353,1.220000e-07,1,19
9,31591.491353,4.700000e-07,1,20
10,31591.491354,2.940000e-07,1,19


In [18]:
import numpy as np

def inspect_npz(npz_path, n=3, k=15):
    d = np.load(npz_path)
    event = d["event"]
    dt    = d["dt"]
    cpu   = d["cpu"]

    print("=== SHAPES ===")
    print("event:", event.shape, event.dtype)
    print("dt   :", dt.shape, dt.dtype)
    print("cpu  :", cpu.shape, cpu.dtype)

    B, L = event.shape

    print("\n=== SAMPLE SEQUENCES (first 20 tokens) ===")
    for i in range(min(n, B)):
        print(f"\n-- seq {i} --")
        print("event:", event[i, :20].tolist())
        print("dt   :", dt[i, :20].tolist())
        print("cpu  :", cpu[i, :20].tolist())

    print("\n=== BASIC STATS ===")
    print("event min/max:", int(event.min()), int(event.max()))
    print("dt    min/max:", int(dt.min()), int(dt.max()))
    print("cpu   min/max:", int(cpu.min()), int(cpu.max()))

    ev_counts = np.bincount(event.reshape(-1))
    top = np.argsort(ev_counts)[::-1][:k]

    print(f"\nTop {k} events:")
    for eid in top:
        if ev_counts[eid] == 0:
            break
        print(f"  event_id={eid}: {int(ev_counts[eid])}")

    cpu_counts = np.bincount(cpu.reshape(-1))
    print("\nCPU distribution:")
    for c in range(len(cpu_counts)):
        if cpu_counts[c] > 0:
            print(f"  cpu {c}: {int(cpu_counts[c])}")

    dt_flat = dt.reshape(-1).astype(np.int64)
    print("\nDT bucket stats:")
    print(
        "  min=", int(dt_flat.min()),
        "max=", int(dt_flat.max()),
        "mean=", float(dt_flat.mean()),
    )

npz_path = r"window_shards\compress-gzip\train\run00_shard0000.npz"
inspect_npz(npz_path, n=5, k=20)

=== SHAPES ===
event: (100000, 200) int32
dt   : (100000, 200) uint8
cpu  : (100000, 200) uint8

=== SAMPLE SEQUENCES (first 20 tokens) ===

-- seq 0 --
event: [19, 33, 34, 23, 20, 18, 18, 25, 19, 20, 19, 117, 1, 55, 8, 8, 95, 23, 23, 115]
dt   : [92, 73, 77, 75, 72, 69, 66, 66, 55, 71, 66, 79, 49, 66, 68, 71, 80, 96, 53, 57]
cpu  : [1, 1, 1, 2, 1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 2, 0, 3, 1]

-- seq 1 --
event: [19, 11, 20, 7, 19, 14, 20, 19, 20, 1, 19, 20, 8, 19, 8, 20, 20, 8, 19, 19]
dt   : [70, 56, 69, 69, 49, 54, 62, 62, 73, 46, 71, 64, 45, 60, 70, 69, 56, 71, 59, 61]
cpu  : [1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 2, 2, 1, 2]

-- seq 2 --
event: [20, 19, 8, 20, 19, 8, 20, 19, 20, 6, 19, 0, 6, 20, 0, 6, 0, 6, 0, 19]
dt   : [53, 72, 58, 44, 62, 59, 64, 73, 70, 71, 36, 57, 61, 53, 35, 59, 54, 56, 54, 35]
cpu  : [1, 1, 2, 1, 1, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1]

-- seq 3 --
event: [20, 10, 13, 19, 11, 14, 20, 19, 19, 7, 20, 8, 19, 20, 158, 158, 8, 8, 11, 7]
dt   : 